In [ ]:
from fastapi import FastAPI, File, UploadFile
from pydantic import BaseModel
from minio import Minio
import json
app = FastAPI()

BUCKET_NAME = "fl-artifacts"

def connect_minio():
    client = Minio(
        endpoint="localhost:9000",
        access_key="admin",
        secret_key="admin12345",
        secure=False
    )
    return client

client = connect_minio()



In [54]:

import sys

import io
sys.path.append(r"D:/DHCN/2025-2026/HK2/CongNgheMoi/project/server")
from model.MLP import MLP
import torch


#load_model_from_minio
def load_fl_params_from_pth_bytes(client_minio):
    model = MLP(input_dim=28*28)

    champion_data = client_minio.get_object(BUCKET_NAME, "registry/champion.json").read().decode('utf-8')
    model_path = json.loads(champion_data)['model_object']
    model_object = client_minio.get_object(BUCKET_NAME, model_path).read()
    

    state_dict = torch.load(io.BytesIO(model_object), map_location="cpu")

    nds = [
        state_dict[k].detach().cpu().numpy()
        for k in sorted(state_dict.keys(), key=lambda x: int(x.split("_")[1]))
    ]
    state_dict = {
            "fc.weight": torch.from_numpy(nds[0]),
            "fc.bias": torch.from_numpy(nds[1])
        }
    model.load_state_dict(state_dict)
    return model
model = load_fl_params_from_pth_bytes(client)


In [ ]:
#download dataset from zenodo
import requests
import os

DATA_DIR = r"D:\DHCN\2025-2026\HK2\CongNgheMoi\project\test_data"
os.makedirs(DATA_DIR, exist_ok=True)

url = "https://zenodo.org/records/10519652/files/pneumoniamnist.npz?download=1"
file_path = os.path.join(DATA_DIR, "pneumoniamnist.npz")

with requests.get(url, stream=True, timeout=10) as r:
    r.raise_for_status()
    with open(file_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

print("Downloaded:", file_path)

Downloaded: D:\DHCN\2025-2026\HK2\CongNgheMoi\project\test_data\pneumoniamnist.npz


In [ ]:
#convert npz to png
import numpy as np
from PIL import Image
import os

NPZ_PATH = r"D:\DHCN\2025-2026\HK2\CongNgheMoi\project\test_data\pneumoniamnist.npz"

SAVE_DIR = r"D:\DHCN\2025-2026\HK2\CongNgheMoi\project\test_images_medmnist"
os.makedirs(SAVE_DIR, exist_ok=True)

data = np.load(NPZ_PATH)

images = data["test_images"]
labels = data["test_labels"]

print("Test images:", images.shape)

for i, (img, label) in enumerate(zip(images, labels)):

    label_name = "pneumonia" if label[0] == 1 else "normal"

    folder = os.path.join(SAVE_DIR, label_name)
    os.makedirs(folder, exist_ok=True)

    image = Image.fromarray(img)

    image.save(os.path.join(folder, f"{label_name}_{i}.png"))

print("Converted all test images")

Test images: (624, 28, 28)
Converted all test images


In [56]:
from PIL import Image
import torch

def preprocess_image(image, normalize=True):
    image = image.convert("L")
    image = image.resize((28, 28))

    tensor = torch.ByteTensor(torch.ByteStorage.from_buffer(image.tobytes()))
    tensor = tensor.float().view(1, 28, 28) / 255.0   # giống ToTensor()

    if normalize:
        tensor = (tensor - 0.5) / 0.5                 # giống Normalize([0.5], [0.5])

    return tensor.view(1, -1)  # flatten thành 784
import os
from PIL import Image
import torch

model.eval()

correct = 0
total = 0

for label_name in ["normal", "pneumonia"]:
    folder = os.path.join(SAVE_DIR, label_name)
    gt = 0 if label_name == "normal" else 1

    for f in os.listdir(folder):
        path = os.path.join(folder, f)
        img = Image.open(path)

        x = preprocess_image(img, normalize=True)

        with torch.no_grad():
            pred = model(x)
            cls = int(pred.item() > 0.5)

        correct += int(cls == gt)
        total += 1

        print(f"{label_name}/{f} -> raw={pred.item():.4f} pred={cls}")

print("accuracy =", correct / total)

normal/normal_1.png -> raw=0.9996 pred=1
normal/normal_103.png -> raw=0.9992 pred=1
normal/normal_106.png -> raw=0.0032 pred=0
normal/normal_107.png -> raw=0.9698 pred=1
normal/normal_109.png -> raw=0.0019 pred=0
normal/normal_112.png -> raw=0.0000 pred=0
normal/normal_114.png -> raw=0.0000 pred=0
normal/normal_118.png -> raw=0.0004 pred=0
normal/normal_119.png -> raw=0.0000 pred=0
normal/normal_123.png -> raw=0.4380 pred=0
normal/normal_125.png -> raw=0.6604 pred=1
normal/normal_126.png -> raw=0.0102 pred=0
normal/normal_128.png -> raw=0.0002 pred=0
normal/normal_131.png -> raw=0.1569 pred=0
normal/normal_133.png -> raw=0.0697 pred=0
normal/normal_136.png -> raw=0.0000 pred=0
normal/normal_139.png -> raw=0.0021 pred=0
normal/normal_142.png -> raw=0.0000 pred=0
normal/normal_143.png -> raw=0.0007 pred=0
normal/normal_144.png -> raw=0.0210 pred=0
normal/normal_145.png -> raw=0.0001 pred=0
normal/normal_146.png -> raw=0.0004 pred=0
normal/normal_147.png -> raw=0.2860 pred=0
normal/normal